In [ ]:
import stac_geoparquet
import geopandas as gpd

from sar_pipeline.utils.stac import read_stac_items_from_s3
from sar_pipeline.utils.aws import S3Util

import tempfile
from pathlib import Path

S3UPLOADER = S3Util()

In [ ]:
bucket = input("Enter S3 bucket name: ") or "dea-public-data-dev"
prefix = input("Enter S3 prefix: ") or "experimental"
item_suffixes = input("Enter item suffixes (comma-separated): ") or "stac-item.json"
item_suffixes = [s.strip() for s in item_suffixes.split(",")]
product_name = input("Enter product name for output file: ") or "ga_s1_nrb"
aws_profile = input("Enter AWS profile name: ")
if aws_profile == "":
    aws_profile = None
num_to_read = input("Enter number of items to read: ")
if num_to_read:
    num_to_read = int(num_to_read)
else:
    num_to_read = None

In [ ]:
loaded_items = read_stac_items_from_s3(
    bucket=bucket,
    prefix=prefix,
    item_suffixes=item_suffixes,
    aws_profile=aws_profile,
    num_to_read=num_to_read,
)

In [ ]:
record_batch_reader = stac_geoparquet.arrow.parse_stac_items_to_arrow(loaded_items)
table = record_batch_reader.read_all()

In [ ]:
parquet_filename = f"{product_name}.parquet"
output_s3_path = f"s3://{bucket}/{prefix}/{parquet_filename}"
print(f"Writing to {output_s3_path}...")

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    local_parquet_path = Path(tmpdir) / parquet_filename
    stac_geoparquet.arrow.to_parquet(table, local_parquet_path)
    S3UPLOADER.s3.put_object(
        Bucket=bucket,
        Key=f"{prefix}/{parquet_filename}",
        Body=local_parquet_path.read_bytes(),
        ContentType="application/octet-stream",
    )

In [ ]:
# gdf = gpd.read_parquet("stac_items.parquet")

In [ ]:
# import geopandas as gpd
# gpd.read_parquet("s3://dea-public-data-dev/experimental/ga_s1_nrb_iw_hh_1.parquet", storage_options={"profile": "dev2"})